# CSI Quality Assurance

This notebook audits the WiCount CSI baseline: timestamp quality, feature definitions, redundancy, PCA, and leakage-safe preparation for future model work. It intentionally stops before ML training.

In [1]:
import json
import zipfile
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

root = Path(r'G:\My Drive\github\projects\WiFiSANSE')
dataset_zip = root / 'analysis' / 'dataset' / 'WiCount' / 'WiCount.zip'
feature_csv = root / 'analysis' / 'processed' / 'features' / 'feature_matrix_50.csv'
output_dir = root / 'analysis' / 'outputs' / 'quality_assurance'
output_dir.mkdir(parents=True, exist_ok=True)
processed_dir = root / 'analysis' / 'processed' / 'features'
processed_dir.mkdir(parents=True, exist_ok=True)
print(f'Root: {root}')
print(f'Output directory: {output_dir}')


Root: G:\My Drive\github\projects\WiFiSANSE
Output directory: G:\My Drive\github\projects\WiFiSANSE\analysis\outputs\quality_assurance


In [2]:
recording_rows = []
with zipfile.ZipFile(dataset_zip) as z:
    files = sorted([n for n in z.namelist() if n.lower().endswith('.csv')])
    for csv_name in files:
        df = pd.read_csv(z.open(csv_name))
        ts = pd.to_datetime(df['timestamp'], errors='coerce')
        intervals = ts.sort_values().diff().dropna()
        recording_rows.append({
            'file': csv_name,
            'recording_id': csv_name.split('/')[-1].replace('.csv', ''),
            'occupancy': int(csv_name.split('/')[1]),
            'rows': int(len(df)),
            'unique_timestamp_count': int(ts.nunique()),
            'timestamp_dup_count': int(ts.duplicated().sum()),
            'zero_interval_count': int((intervals == pd.Timedelta(0)).sum()),
            'negative_interval_count': int((intervals < pd.Timedelta(0)).sum()),
            'min_interval_ms': float(intervals.min().total_seconds() * 1000) if len(intervals) else np.nan,
            'median_interval_ms': float(intervals.median().total_seconds() * 1000) if len(intervals) else np.nan,
            'max_interval_ms': float(intervals.max().total_seconds() * 1000) if len(intervals) else np.nan,
        })

recording_summary = pd.DataFrame(recording_rows).sort_values('occupancy').reset_index(drop=True)
recording_summary.to_csv(processed_dir / 'timestamp_quality_summary.csv', index=False)
print('Total recordings:', len(recording_summary))
print('Total rows:', recording_summary['rows'].sum())
print('Total duplicate timestamps:', recording_summary['timestamp_dup_count'].sum())
print('Total zero intervals:', recording_summary['zero_interval_count'].sum())
print('Total negative intervals:', recording_summary['negative_interval_count'].sum())
print(recording_summary[['recording_id','occupancy','rows','unique_timestamp_count','timestamp_dup_count','zero_interval_count','negative_interval_count']].to_string(index=False))


Total recordings: 15
Total rows: 78618
Total duplicate timestamps: 8900
Total zero intervals: 8900
Total negative intervals: 0
                 recording_id  occupancy  rows  unique_timestamp_count  timestamp_dup_count  zero_interval_count  negative_interval_count
2023-12-21_19-23-53-985_104_1          0  5218                    3877                 1341                 1341                        0
2023-12-21_19-27-21-930_104_1          1  4780                    3935                  845                  845                        0
2023-12-21_19-31-17-706_104_1          1  5189                    4196                  993                  993                        0
2023-12-21_19-34-42-783_104_1          1  5234                    5052                  182                  182                        0
2023-12-21_19-37-36-861_104_1          1  5163                    4211                  952                  952                        0
2023-12-21_19-40-12-110_104_1          1  542

In [3]:
feature_df = pd.read_csv(feature_csv)
meta_cols = ['window_id', 'recording_id', 'occupancy', 'window_size', 'start_index', 'end_index', 'row_count']
numeric_df = feature_df.drop(columns=meta_cols).replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)
feature_df_clean = feature_df.copy()
feature_df_clean[numeric_df.columns] = numeric_df

std_equal = []
for sub in range(52):
    s1 = f'sub_{sub}_std'
    s2 = f'sub_{sub}_temporal_std'
    if s1 in numeric_df.columns and s2 in numeric_df.columns:
        diff = np.max(np.abs(numeric_df[s1].to_numpy() - numeric_df[s2].to_numpy()))
        std_equal.append({'subcarrier': sub, 'max_abs_diff': float(diff), 'identical': bool(diff == 0.0)})

std_equal_df = pd.DataFrame(std_equal)
print('std == temporal_std count:', int(std_equal_df['identical'].sum()))
print('max abs diff between std and temporal_std:', float(std_equal_df['max_abs_diff'].max()))

# Feature redundancy analysis
corr = numeric_df.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr_pairs = upper.stack().reset_index()
high_corr_pairs.columns = ['feature_a', 'feature_b', 'corr']
high_corr_pairs = high_corr_pairs[high_corr_pairs['corr'] >= 0.9].sort_values('corr', ascending=False).reset_index(drop=True)
high_corr_pairs.to_csv(processed_dir / 'high_correlation_pairs.csv', index=False)

var_series = numeric_df.var(ddof=1)
constant_features = var_series[var_series <= 1e-12].index.tolist()
low_variance = var_series.sort_values().head(10).reset_index()
low_variance.columns = ['feature', 'variance']
low_variance.to_csv(processed_dir / 'low_variance_features.csv', index=False)

summary = {
    'n_window_rows': int(feature_df_clean.shape[0]),
    'n_features': int(numeric_df.shape[1]),
    'n_metadata': int(len(meta_cols)),
    'n_nan': int(feature_df_clean.isna().sum().sum()),
    'n_inf': int(np.isinf(numeric_df.to_numpy(dtype=float)).sum()),
    'n_high_corr_pairs': int(len(high_corr_pairs)),
    'n_constant_features': int(len(constant_features)),
    'n_features_var_le_1e12': int((var_series <= 1e-12).sum()),
    'std_temporal_std_identical': bool(std_equal_df['identical'].all()),
}
summary_df = pd.DataFrame([summary])
summary_df.to_csv(processed_dir / 'feature_quality_summary.csv', index=False)
print(summary_df.to_string(index=False))
print('Top redundant feature pairs:')
print(high_corr_pairs.head(10).to_string(index=False))
print('Low variance features:')
print(low_variance.to_string(index=False))


std == temporal_std count: 52
max abs diff between std and temporal_std: 0.0


 n_window_rows  n_features  n_metadata  n_nan  n_inf  n_high_corr_pairs  n_constant_features  n_features_var_le_1e12  std_temporal_std_identical
          3122         992           7      0      0               3892                    0                       0                        True
Top redundant feature pairs:
 feature_a           feature_b  corr
sub_50_std sub_50_temporal_std   1.0
 sub_0_std  sub_0_temporal_std   1.0
sub_47_std sub_47_temporal_std   1.0
sub_46_std sub_46_temporal_std   1.0
sub_48_std sub_48_temporal_std   1.0
sub_49_std sub_49_temporal_std   1.0
sub_15_std sub_15_temporal_std   1.0
sub_14_std sub_14_temporal_std   1.0
sub_12_std sub_12_temporal_std   1.0
sub_20_std sub_20_temporal_std   1.0
Low variance features:
               feature  variance
sub_51_first_diff_mean  0.002380
sub_50_first_diff_mean  0.002760
sub_49_first_diff_mean  0.002944
sub_47_first_diff_mean  0.003250
sub_48_first_diff_mean  0.003260
sub_46_first_diff_mean  0.003275
sub_45_first_diff_me

In [4]:
# PCA with numeric-only features and z-score standardization
X = numeric_df.to_numpy(dtype=float)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
X_centered = X - X.mean(axis=0)
X_scale = X.std(axis=0, ddof=1)
X_scale[X_scale == 0] = 1.0
X_std = X_centered / X_scale
U, S, Vt = np.linalg.svd(X_std, full_matrices=False)
explained_var = (S ** 2) / (X_std.shape[0] - 1)
explained_var_ratio = explained_var / explained_var.sum()
cumulative = np.cumsum(explained_var_ratio)
pca_summary = pd.DataFrame({
    'component': np.arange(1, len(explained_var) + 1),
    'explained_variance': explained_var,
    'explained_variance_ratio': explained_var_ratio,
    'cumulative_explained_variance': cumulative,
})
pca_summary.to_csv(processed_dir / 'pca_explained_variance_summary.csv', index=False)
print('PCA first 5 ratios:', np.round(explained_var_ratio[:5].tolist(), 6).tolist())
print('PCA cumulative first 5:', np.round(cumulative[:5].tolist(), 6).tolist())
print('PCA 90% threshold components:', int(np.argmax(cumulative >= 0.90) + 1))
print('PCA 95% threshold components:', int(np.argmax(cumulative >= 0.95) + 1))


PCA first 5 ratios: [0.275024, 0.122064, 0.089492, 0.077872, 0.050813]
PCA cumulative first 5: [0.275024, 0.397088, 0.48658, 0.564452, 0.615265]
PCA 90% threshold components: 39
PCA 95% threshold components: 71


In [5]:
# Representative heatmap for a subset of the feature matrix
subset_cols = [
    'sub_0_mean', 'sub_0_std', 'sub_0_rms', 'sub_0_energy',
    'sub_1_mean', 'sub_1_std', 'sub_1_rms', 'sub_1_energy',
    'sub_2_mean', 'sub_2_std', 'sub_2_rms', 'sub_2_energy',
    'sub_3_mean', 'sub_3_std', 'sub_3_rms', 'sub_3_energy',
    'sub_4_mean', 'sub_4_std', 'sub_4_rms', 'sub_4_energy',
]
subset = numeric_df[subset_cols]
heatmap = subset.corr()
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(heatmap.values, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(subset_cols)))
ax.set_yticks(range(len(subset_cols)))
ax.set_xticklabels(subset_cols, rotation=45, ha='right')
ax.set_yticklabels(subset_cols)
ax.set_title('Feature correlation heatmap (representative subset)')
fig.colorbar(im, ax=ax, label='Pearson correlation')
fig.tight_layout()
fig.savefig(output_dir / 'feature_correlation_heatmap.png', dpi=200)
plt.close(fig)
print('Saved feature correlation heatmap to', output_dir / 'feature_correlation_heatmap.png')


Saved feature correlation heatmap to G:\My Drive\github\projects\WiFiSANSE\analysis\outputs\quality_assurance\feature_correlation_heatmap.png


In [6]:
# Feature variance distribution plot
var_series = numeric_df.var(ddof=1)
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(var_series, bins=50, color='steelblue', alpha=0.8)
ax.axvline(1e-12, color='red', linestyle='--', label='near-zero threshold')
ax.set_title('Feature variance distribution')
ax.set_xlabel('Variance')
ax.set_ylabel('Feature count')
ax.legend()
fig.tight_layout()
fig.savefig(output_dir / 'feature_variance_distribution.png', dpi=200)
plt.close(fig)
print('Saved feature variance distribution')


Saved feature variance distribution


In [7]:
# High-correlation pair ranking plot
pair_plot = high_corr_pairs.head(20).copy()
pair_plot['label'] = pair_plot['feature_a'] + ' / ' + pair_plot['feature_b']
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(pair_plot['label'][::-1], pair_plot['corr'][::-1], color='darkorange')
ax.set_title('Top high-correlation feature pairs (|r| >= 0.90)')
ax.set_xlabel('Absolute Pearson correlation')
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(output_dir / 'high_correlation_pairs.png', dpi=200)
plt.close(fig)
print('Saved high-correlation pairs plot')


Saved high-correlation pairs plot


In [8]:
# PCA explained variance plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(pca_summary['component'][:20], pca_summary['explained_variance_ratio'][:20], marker='o', label='Variance ratio', color='tab:blue')
ax.plot(pca_summary['component'][:20], pca_summary['cumulative_explained_variance'][:20], marker='s', label='Cumulative variance', color='tab:orange')
ax.set_title('PCA explained variance (first 20 components)')
ax.set_xlabel('Principal component')
ax.set_ylabel('Variance ratio')
ax.legend()
fig.tight_layout()
fig.savefig(output_dir / 'pca_explained_variance.png', dpi=200)
plt.close(fig)
print('Saved PCA explained variance plot')


Saved PCA explained variance plot


In [9]:
# PCA scatter plot
score_matrix = X_std @ Vt[:2].T
pc1 = score_matrix[:, 0]
pc2 = score_matrix[:, 1]
occupancy_values = feature_df_clean['occupancy'].to_numpy()
fig, ax = plt.subplots(figsize=(8, 6))
for occ in sorted(np.unique(occupancy_values)):
    mask = occupancy_values == occ
    ax.scatter(pc1[mask], pc2[mask], s=12, alpha=0.7, label=f'occupancy {occ}')
ax.set_title('PCA scatter plot (PC1 vs PC2)')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend()
fig.tight_layout()
fig.savefig(output_dir / 'pca_scatter.png', dpi=200)
plt.close(fig)
print('Saved PCA scatter plot')


Saved PCA scatter plot


In [10]:
# Timestamp interval distribution and recording quality summary
all_intervals = []
with zipfile.ZipFile(dataset_zip) as z:
    for csv_name in sorted([n for n in z.namelist() if n.lower().endswith('.csv')]):
        df = pd.read_csv(z.open(csv_name))
        ts = pd.to_datetime(df['timestamp'], errors='coerce')
        intervals = ts.sort_values().diff().dropna()
        all_intervals.extend(intervals.dt.total_seconds().to_list())

intervals_s = pd.Series(all_intervals)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(intervals_s, bins=40, color='forestgreen', alpha=0.8)
axes[0].set_title('Timestamp interval distribution (seconds)')
axes[0].set_xlabel('Interval (s)')
axes[0].set_ylabel('Count')
axes[1].bar(recording_summary['recording_id'], recording_summary['timestamp_dup_count'], color='tomato')
axes[1].set_title('Duplicate timestamp count by recording')
axes[1].set_xlabel('Recording ID')
axes[1].set_ylabel('Duplicate timestamps')
axes[1].tick_params(axis='x', rotation=45)
fig.tight_layout()
fig.savefig(output_dir / 'timestamp_interval_distribution.png', dpi=200)
plt.close(fig)
print('Saved timestamp interval distribution plot')

recording_summary['timestamp_dup_fraction'] = recording_summary['timestamp_dup_count'] / recording_summary['rows']
recording_summary[['recording_id', 'occupancy', 'rows', 'timestamp_dup_count', 'timestamp_dup_fraction']].to_csv(processed_dir / 'recording_quality_summary.csv', index=False)
print(recording_summary[['recording_id', 'occupancy', 'rows', 'timestamp_dup_count']].to_string(index=False))


Saved timestamp interval distribution plot
                 recording_id  occupancy  rows  timestamp_dup_count
2023-12-21_19-23-53-985_104_1          0  5218                 1341
2023-12-21_19-27-21-930_104_1          1  4780                  845
2023-12-21_19-31-17-706_104_1          1  5189                  993
2023-12-21_19-34-42-783_104_1          1  5234                  182
2023-12-21_19-37-36-861_104_1          1  5163                  952
2023-12-21_19-40-12-110_104_1          1  5421                 1147
2023-12-21_19-42-28-267_104_1          1  5644                 1264
2023-12-21_19-45-07-501_104_1          1  5128                  359
2023-12-21_19-47-14-669_104_1          2  5376                    0
2023-12-21_19-49-37-940_104_1          2  5307                 1073
2023-12-21_19-54-01-269_104_1          2  5141                  186
2023-12-21_19-56-22-903_104_1          2  5467                  558
2023-12-21_20-02-00-329_104_1          2  5221                    0
2023-

In [11]:
# Final summary for the report
print('=== QUALITY ASSURANCE SUMMARY ===')
print(f'Processed recordings: {len(recording_summary)}')
print(f'Total rows: {recording_summary["rows"].sum()}')
print(f'Total duplicate timestamps: {recording_summary["timestamp_dup_count"].sum()}')
print(f'Total zero intervals: {recording_summary["zero_interval_count"].sum()}')
print(f'Total negative intervals: {recording_summary["negative_interval_count"].sum()}')
print(f'Feature matrix shape: {feature_df_clean.shape}')
print(f'Numeric features: {numeric_df.shape[1]}')
print(f'High-correlation pairs (|r| >= 0.90): {len(high_corr_pairs)}')
print(f'Constant features: {len(constant_features)}')
print(f'PCA first component ratio: {explained_var_ratio[0]:.4f}')
print(f'PCA cumulative first 5: {cumulative[:5].tolist()}')


=== QUALITY ASSURANCE SUMMARY ===
Processed recordings: 15
Total rows: 78618
Total duplicate timestamps: 8900
Total zero intervals: 8900
Total negative intervals: 0
Feature matrix shape: (3122, 999)
Numeric features: 992
High-correlation pairs (|r| >= 0.90): 3892
Constant features: 0
PCA first component ratio: 0.2750
PCA cumulative first 5: [0.2750235932261145, 0.3970877586004811, 0.48658013215182055, 0.5644518725249715, 0.6152649552441237]
